# 03 - Trays, Modules, And Filters

Build a small IceTray processing chain. A tray reads frames, modules inspect or modify them, and writer modules can save selected output.


In [ ]:
from pathlib import Path
import os
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

from icecube import icetray, dataclasses, dataio
from I3Tray import I3Tray

from icetray_tutorial.paths import EXPERIMENT
from icetray_tutorial.filters import filter_passed
from icetray_tutorial.pulses import find_pulse_key


In [ ]:
def add_hit_dom_count(frame, pulse_key=None):
    key = pulse_key or find_pulse_key(frame)
    if key is None:
        return True
    pulses = dataclasses.I3RecoPulseSeriesMap.from_frame(frame, key)
    hit_doms = sum(1 for _, series in pulses if len(series) > 0)
    frame['TutorialHitDOMs'] = icetray.I3Int(hit_doms)
    return True

def keep_minimum_hit_doms(frame, minimum_doms=20):
    return 'TutorialHitDOMs' in frame and int(frame['TutorialHitDOMs'].value) >= minimum_doms

def keep_filter(frame, filter_name='MuonFilter_13'):
    return filter_passed(frame, filter_name)


In [ ]:
username = os.environ.get('USER', 'icecube_user')
output_dir = Path(f'/data/user/{username}/IceTray_tutorial')
output_dir.mkdir(parents=True, exist_ok=True)
selected_i3 = output_dir / 'selected_min20doms.i3.zst'
selected_i3


In [ ]:
tray = I3Tray()
tray.AddModule('I3Reader', 'reader', FilenameList=[str(EXPERIMENT.gcd), str(EXPERIMENT.event_file)])
tray.AddModule(add_hit_dom_count, 'add_hit_dom_count', Streams=[icetray.I3Frame.Physics])
tray.AddModule(keep_minimum_hit_doms, 'keep_minimum_hit_doms', minimum_doms=20, Streams=[icetray.I3Frame.Physics])
tray.AddModule('I3Writer', 'writer', Filename=str(selected_i3), Streams=[icetray.I3Frame.TrayInfo, icetray.I3Frame.DAQ, icetray.I3Frame.Physics])
tray.Execute(200)
tray.Finish()
selected_i3


## Practice

1. Change the DOM threshold and compare output size.
2. Add `keep_filter` to select events using `QFilterMask`.
3. Write an `I3Bool` key recording whether a specific filter passed.
